# RecursionLens - JupyterLab tutorial

**This notebook is pre-executed.** Every code cell already contains saved output. If Jupyter hides the HTML/SVG because the downloaded notebook is untrusted, use the notebook's **Trust** action (shield icon), or select **Run All Cells** once. The static HTML companion always shows every report without trusting or running anything.

In this tutorial you will:

1. trace a normal recursive function without changing its return value;
2. compare a collapsible call tree with a fully expanded SVG tree;
3. merge repeated subproblems into a DAG;
4. measure what memoization changes;
5. export standalone evidence.

> **To rerun it yourself:** select **Kernel -> Restart Kernel and Run All Cells**. No Graphviz, native library, external JavaScript, or CDN is required.

This file targets **RecursionLens 0.1.2**.

## Setup

Install in the same kernel that runs this notebook.

After the PyPI release:

```python
%pip install recursionlens
```

Before the PyPI release, install the tagged GitHub version:

```python
%pip install "git+https://github.com/ZiyuJiang07/recursionlens.git@v0.1.2"
```

If Jupyter reports that the package is missing immediately after installation, restart the kernel once. The next cell is already executed and shows what a successful import looks like.

In [1]:
import recursionlens
from recursionlens import trace

print("RecursionLens version:", recursionlens.__version__)

RecursionLens version: 0.1.2


## 1. Trace a function without changing it

`@trace` records calls while leaving the function's normal behaviour intact. A normal call still returns the original value and raises the original exception.

The function below is the naive Fibonacci implementation. It deliberately recomputes the same subproblems many times.

In [2]:
@trace
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

value = fib(5)
assert value == 5
print("Return value:", value)
print("Calls recorded by the normal call:", fib.last_report.total_calls)

Return value: 5
Calls recorded by the normal call: 15


## 2. Collapsible call tree

A report object renders automatically at the end of a JupyterLab cell. This view is useful for large runs: no node is open by default beyond the first levels, and each triangle expands its children.

Watch for:

- **repeated**: the same argument signature has already appeared;
- **cache hit**: a memoized call reused a result;
- **error**: the invocation raised an exception.

In [3]:
naive = fib.run(5)
naive

TraceReport(function_name='fib', value=5, nodes=[CallNode(call_id=0, identity='1:__main__:fib(n:builtins.int=5)', signature='fib(n=5)', depth=1, status='returned', result='5', error=None, repeated=False, children=[CallNode(call_id=1, identity='1:__main__:fib(n:builtins.int=4)', signature='fib(n=4)', depth=2, status='returned', result='3', error=None, repeated=False, children=[CallNode(call_id=2, identity='1:__main__:fib(n:builtins.int=3)', signature='fib(n=3)', depth=3, status='returned', result='2', error=None, repeated=False, children=[CallNode(call_id=3, identity='1:__main__:fib(n:builtins.int=2)', signature='fib(n=2)', depth=4, status='returned', result='1', error=None, repeated=False, children=[CallNode(call_id=4, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=5, status='returned', result='1', error=None, repeated=False, children=[]), CallNode(call_id=5, identity='1:__main__:fib(n:builtins.int=0)', signature='fib(n=0)', depth=5, status='returned', result='0', error=None, repeated=False, children=[])]), CallNode(call_id=6, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=4, status='returned', result='1', error=None, repeated=True, children=[])]), CallNode(call_id=7, identity='1:__main__:fib(n:builtins.int=2)', signature='fib(n=2)', depth=3, status='returned', result='1', error=None, repeated=True, children=[CallNode(call_id=8, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=4, status='returned', result='1', error=None, repeated=True, children=[]), CallNode(call_id=9, identity='1:__main__:fib(n:builtins.int=0)', signature='fib(n=0)', depth=4, status='returned', result='0', error=None, repeated=True, children=[])])]), CallNode(call_id=10, identity='1:__main__:fib(n:builtins.int=3)', signature='fib(n=3)', depth=2, status='returned', result='2', error=None, repeated=True, children=[CallNode(call_id=11, identity='1:__main__:fib(n:builtins.int=2)', signature='fib(n=2)', depth=3, status='returned', result='1', error=None, repeated=True, children=[CallNode(call_id=12, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=4, status='returned', result='1', error=None, repeated=True, children=[]), CallNode(call_id=13, identity='1:__main__:fib(n:builtins.int=0)', signature='fib(n=0)', depth=4, status='returned', result='0', error=None, repeated=True, children=[])]), CallNode(call_id=14, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=3, status='returned', result='1', error=None, repeated=True, children=[])])]), CallNode(call_id=1, identity='1:__main__:fib(n:builtins.int=4)', signature='fib(n=4)', depth=2, status='returned', result='3', error=None, repeated=False, children=[CallNode(call_id=2, identity='1:__main__:fib(n:builtins.int=3)', signature='fib(n=3)', depth=3, status='returned', result='2', error=None, repeated=False, children=[CallNode(call_id=3, identity='1:__main__:fib(n:builtins.int=2)', signature='fib(n=2)', depth=4, status='returned', result='1', error=None, repeated=False, children=[CallNode(call_id=4, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=5, status='returned', result='1', error=None, repeated=False, children=[]), CallNode(call_id=5, identity='1:__main__:fib(n:builtins.int=0)', signature='fib(n=0)', depth=5, status='returned', result='0', error=None, repeated=False, children=[])]), CallNode(call_id=6, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=4, status='returned', result='1', error=None, repeated=True, children=[])]), CallNode(call_id=7, identity='1:__main__:fib(n:builtins.int=2)', signature='fib(n=2)', depth=3, status='returned', result='1', error=None, repeated=True, children=[CallNode(call_id=8, identity='1:__main__:fib(n:builtins.int=1)', signature='fib(n=1)', depth=4, status='returned', result='1', error=None, repeated=True, children=[]), CallNode(call_id=9, identity='1:__main__:fib(n:builtins.int=0)', signature='fib(n=0)', depth=4, s

## 3. Fully expanded SVG tree

`mode="expanded"` shows every recorded invocation at once. This is the view to use when explaining the complete recursion shape.

- black labels: first occurrence of a subproblem;
- underlined coral labels: repeated calls;
- green labels: cache hits;
- red labels: errors.

For `fib(5)`, all **15** calls are visible without clicking.

In [4]:
naive.show(mode="expanded")

## 4. The same run as a subproblem DAG

The tree contains every invocation. The DAG merges invocations with identical arguments into one node, so **15 calls collapse into 6 unique subproblems**.

The DAG keeps dependency edges and stores three counts per node:

- `calls`: how many times the subproblem was observed;
- `executed`: how many times its body actually ran;
- `cached`: how many times its result came from the cache.

In [5]:
assert len(naive.dag_nodes) == 6
assert len(naive.dag_edges) == 8
print(f"DAG: {len(naive.dag_nodes)} nodes, {len(naive.dag_edges)} edges")
naive.show(mode="dag")

DAG: 6 nodes, 8 edges


## 5. Compare naive and memoized execution

The memoized run calls the same function with the same input, but caches results by argument identity.

A cache probe still counts as an **observed call**, but it does not execute the function body. That is why the memoized run observes 9 calls while executing only 6 bodies.

In [6]:
memoized = fib.with_options(memoize=True).run(5)
assert memoized.value == naive.value == 5
assert memoized.executed_calls == 6
assert memoized.cached_calls == 3

comparison = naive.compare(memoized)
comparison

Metric,naive,memoized,Change
Observed calls,15,9,-6
Executed bodies,15,6,-9
Cache hits,0,3,+3
Unique subproblems,6,6,+0
Repeated calls,9,3,-6
Maximum depth,5,5,+0


## 6. The numbers to remember

| Metric | Naive | Memoized | Why |
|---|---:|---:|---|
| Observed calls | 15 | 9 | Cache probes are still calls |
| Executed bodies | 15 | 6 | Only one body per unique subproblem |
| Cache hits | 0 | 3 | `fib(1)`, `fib(2)`, `fib(3)` reused |
| Unique subproblems | 6 | 6 | Same problem space |
| DAG edges | 8 | 8 | Same dependency structure |
| Maximum depth | 5 | 5 | Recursion chain is unchanged |

The headline result is **15 executed bodies -> 6 executed bodies**, not 15 -> 9. The 9 includes cache probes.

## 7. Contrast with linear recursion

Repeated subproblems are not inevitable. Factorial has deep recursion but no repeated arguments, so its DAG has one node per call and **zero repeated calls**.

In [7]:
@trace
def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)

linear = factorial.run(5)
assert linear.value == 120
assert linear.repeated_calls == 0
print("factorial(5):", linear.value)
print("calls:", linear.total_calls, "unique:", linear.unique_subproblems, "repeated:", linear.repeated_calls)
linear.show(mode="expanded")

factorial(5): 120
calls: 5 unique: 5 repeated: 0


## 8. Export evidence

Reports can leave the notebook as standalone HTML or versioned JSON. The following cell writes all three views plus one JSON trace to a temporary directory and prints their sizes.

In [8]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as folder:
    output = Path(folder)
    naive.export_html(output / "fib-tree.html")
    naive.export_html(output / "fib-expanded.html", mode="expanded")
    naive.export_html(output / "fib-dag.html", mode="dag")
    naive.export_json(output / "fib-trace.json")
    for path in sorted(output.iterdir()):
        print(f"{path.name:22} {path.stat().st_size:,} bytes")

fib-dag.html           6,841 bytes
fib-expanded.html      9,041 bytes
fib-trace.json         8,571 bytes
fib-tree.html          5,657 bytes


## Troubleshooting

### The cell shows text instead of a rendered tree
Put the report on the last line of a code cell, call `report.show()`, or use `report.show(mode="expanded")`. Outside IPython, `show()` returns the HTML string instead of displaying it.

### A large tree is truncated
The default limit is 1,000 recorded nodes. Raise it deliberately with `fib.with_options(max_calls=5000).run(...)`. The actual function result is never truncated.

### Memoization did not apply
Memoization requires hashable bound arguments. Unhashable arguments produce a warning and those calls remain uncached.

### The report says an exception was raised
RecursionLens records the error and re-raises the original exception. Inspect the last run with `function.last_report`.

### Scope
Version 0.1 targets synchronous, deterministic teaching functions. It is not a sandbox for untrusted code, and repeated arguments do not imply redundant work for side-effecting functions.

## One-minute demo recipe

1. Run the naive Fibonacci cell and say: "15 calls, but only 6 unique subproblems."
2. Show the expanded tree for five seconds.
3. Show the DAG and say: "This is the recursion after merging identical work."
4. Show the comparison and say: "Memoization executes 6 bodies instead of 15."
5. Close with the innovation: "One trace becomes a call tree, a subproblem DAG, and a measured comparison without Graphviz."

Next: [README](https://github.com/ZiyuJiang07/recursionlens#readme), [User guide](https://github.com/ZiyuJiang07/recursionlens/blob/main/docs/USER.md), [Example gallery](https://github.com/ZiyuJiang07/recursionlens/tree/main/examples).